# HDFS Fault Tolerance Scenarios — Interactive Lab

Lab 04 showed how HDFS **auto-heals** after a DataNode dies. This lab goes further into real-world operational scenarios that cluster administrators face:

1. **Safe Mode** — HDFS read-only mode on startup (and how to manage it)
2. **Block Rebalancing** — moving blocks evenly after adding a node
3. **Graceful Decommissioning** — retiring a DataNode without data loss
4. **NameNode HA** — how production clusters eliminate the single point of failure

> Some steps run inside cluster containers via the `hadoop()` helper (same pattern as labs 04–06).

## Setup

In [ ]:
import sys; sys.path.insert(0, '../scripts')
from hdfs_utils import hadoop, compose, block_report, safe_delete, docker_cp
from hdfs import InsecureClient

client = InsecureClient('http://localhost:14000', user='root')
print('Connected to HDFS via proxy ✓')

## 1. Upload test data

We need enough data to see meaningful block distribution. If you already have data from lab 04 (`/datasets/nyc_taxi`), we reuse it. Otherwise we download a small sample.

In [ ]:
TEST_DIR = '/datasets/ftest'

# Create small demo files with different block sizes to populate the cluster
import os
os.makedirs('../temp/ftest', exist_ok=True)

# Generate a ~200 MB file in chunks so we get multiple blocks
chunk_path = '../temp/ftest/200mb.dat'
if not os.path.exists(chunk_path):
    print('Generating 200 MB test file (one-time)...')
    with open(chunk_path, 'wb') as f:
        for i in range(200):
            f.write(b'x' * (1024 * 1024))  # 1 MB at a time
    print(f'Generated {chunk_path} ({os.path.getsize(chunk_path) / 1e6:.0f} MB)')

client.makedirs(TEST_DIR, permission='755')
client.upload(f'{TEST_DIR}/200mb.dat', chunk_path, overwrite=True)
print(f'Uploaded to HDFS: {TEST_DIR}/200mb.dat')

# Check how many blocks it created
print('\nBlock report:')
report = hadoop(f'hdfs fsck {TEST_DIR}/200mb.dat -files -blocks -locations | head -20')
print(report)

## 2. Safe Mode — HDFS Read-Only Mode

When the NameNode starts, it enters **Safe Mode** — a read-only state where:
- Blocks are **not replicated** (no clients can write)
- The NameNode collects heartbeats from DataNodes
- Safe Mode exits when >= `dfs.namenode.safemode.threshold-pct` of blocks (default 99.9%) are reported

### Why this matters
- After a cluster-wide failure, you may need to **manually enter/leave safe mode** for maintenance
- A corrupt cluster can get stuck in safe mode — knowing how to diagnose and fix it is critical

### CLI equivalents
```bash
hdfs dfsadmin -safemode get      # check current state
hdfs dfsadmin -safemode enter    # force safe mode
hdfs dfsadmin -safemode leave    # force exit safe mode
hdfs dfsadmin -safemode wait     # block until safe mode exits
```

In [ ]:
# 2a. Check current safe mode state
current = hadoop('hdfs dfsadmin -safemode get')
print(current)
is_safe = 'ON' in current
print('\n→ Cluster is', 'IN safe mode' if is_safe else 'OUT of safe mode')

In [ ]:
# 2b. Force enter safe mode (writes will be rejected)
print('Forcing safe mode...')
result = hadoop('hdfs dfsadmin -safemode enter')
print(result)

# Verify: try to write a file — this should fail
print('\nAttempting to write while in safe mode...')
try:
    with client.write(f'{TEST_DIR}/safe_mode_test.txt', overwrite=True) as f:
        f.write(b'This should fail!')
    print('  WROTE SUCCESSFULLY (unexpected)')
except Exception as e:
    print(f'  Write rejected as expected: {type(e).__name__}: {e}')

In [ ]:
# 2c. Leave safe mode
print('Leaving safe mode...')
result = hadoop('hdfs dfsadmin -safemode leave')
print(result)

# Verify: write should work again
print('\nAttempting to write after leaving safe mode...')
with client.write(f'{TEST_DIR}/safe_mode_test.txt', overwrite=True) as f:
    f.write(b'This should work!')
print('  Write succeeded ✓')

# Clean up
client.delete(f'{TEST_DIR}/safe_mode_test.txt')

## 3. Block Rebalancing

When a new DataNode is added to a running cluster, it starts empty — all existing data stays on the old nodes. **`hdfs balancer`** moves blocks so that every node has roughly equal disk usage.

### Scenario
Lab 04 killed and restarted `datanode3`. When a DataNode restarts, it comes back empty (in the old `init-datanode.sh`). The new version preserves data, but let's simulate adding a fresh empty node to see balancing in action.

### CLI equivalent
```bash
hdfs balancer -threshold 10    # rebalance until each node is within 10%% of average
```

In [ ]:
# 3a. Check current per-node storage balance
report = hadoop('hdfs dfsadmin -report')
# Extract DFS Used% for each DataNode (track the current host as we scan;
# skip the cluster-wide summary lines that appear before any Hostname:)
host = None
for line in report.split('\n'):
    line = line.strip()
    if line.startswith('Hostname:'):
        host = line.split(':', 1)[-1].strip()
    elif host and (line.startswith('DFS Used%:') or line.startswith('DFS Remaining%:')):
        print(f'  {host}: {line}')

In [ ]:
# 3b. Simulate an imbalanced cluster: stop a DataNode, then restart it
#    After restart it has no blocks (in a real decommission scenario).
print('Stopping datanode3 to simulate a fresh node...')
print(compose('stop datanode3'))
import time
time.sleep(5)
print('\nStarting datanode3 back...')
print(compose('start datanode3'))
time.sleep(15)  # wait for registration

print('\n--- Re-checking balance ---')
report = hadoop('hdfs dfsadmin -report')
host = None
for line in report.split('\n'):
    line = line.strip()
    if line.startswith('Hostname:'):
        host = line.split(':', 1)[-1].strip()
    elif host and (line.startswith('DFS Used%:') or line.startswith('DFS Remaining%:')):
        print(f'  {host}: {line}')

In [ ]:
# 3c. Run the HDFS Balancer
#    The balancer moves blocks from over-utilized to under-utilized DataNodes.
#    We use a 10% threshold (nodes within 10% of avg utilization are considered balanced).
print('Running HDFS Balancer (threshold=10)...')
print('(This may take a minute depending on data volume)')
balance_log = hadoop('hdfs balancer -threshold 10', timeout=300)
# Show the summary lines at the end
summary = balance_log[-1500:]
print(summary)

In [ ]:
# 3d. Verify balance improved
print('--- Post-balance DataNode usage ---')
report = hadoop('hdfs dfsadmin -report')
host = None
usages = []
for line in report.split('\n'):
    line = line.strip()
    if line.startswith('Hostname:'):
        host = line.split(':', 1)[-1].strip()
    elif host and line.startswith('DFS Used%:'):
        pct = line.split(':', 1)[-1].strip()
        usages.append((host, pct))
        print(f'  {host}: {pct}')

if len(usages) >= 3:
    values = [float(u[1].rstrip('%')) for u in usages]
    spread = max(values) - min(values)
    print(f'\nUsage spread: {spread:.1f}% (lower = more balanced)')
    print('The balancer reduced the gap compared to the pre-balance state.')

## 4. Graceful Decommissioning

In production, you don't just `kill` a DataNode. You **decommission** it — telling the NameNode to replicate its blocks to other nodes before shutting it down. This ensures zero data loss.

### Scenario
We mark `datanode3` for decommission. The NameNode copies its blocks to the remaining DataNodes. Once replication is complete, we can safely remove the node.

### CLI equivalents
```bash
# Mark for decommission (via dfsadmin or config file)
hdfs dfsadmin -refreshNodes

# Check decommission status
hdfs dfsadmin -report

# Recommission (cancel decommission)
# Remove from exclude file and refresh
hdfs dfsadmin -refreshNodes
```

In [ ]:
# 4a. Check current live/dead DataNodes
report = hadoop('hdfs dfsadmin -report 2>&1 | grep -E "(Live|Dead|Decommission) datanodes"')
print(report)

In [ ]:
# 4b. Create a decommission exclude file and refresh nodes
#    The exclude file tells the NameNode which DataNodes to decommission.

# Write exclude file locally, then copy to namenode container
exclude_content = 'datanode3'
local_exclude = '../temp/ftest/exclude-hosts'
with open(local_exclude, 'w') as f:
    f.write(exclude_content)

# Copy into container
docker_cp(local_exclude, '/opt/hadoop/etc/hadoop/exclude-hosts')

# Tell the NameNode to load the exclude file
result = hadoop('hdfs dfsadmin -refreshNodes')
print('RefreshNodes result:')
print(result)

In [ ]:
# 4c. Watch decommission progress
#    Blocks on datanode3 are being replicated to other nodes.
#    Once all blocks are replicated (Decommission Status = Decommissioned), the node is safe to remove.
print('Checking decommission status (may take a moment)...')
import time
for attempt in range(6):
    report = hadoop('hdfs dfsadmin -report 2>&1 | grep -A5 "datanode3"')
    if 'Decommission in progress' in report or 'Decommissioned' in report:
        print(f'Attempt {attempt+1}: decommission active')
        for line in report.split('\n'):
            if any(x in line for x in ['Hostname:', 'Decommission', 'DFS Used%']):
                print(f'  {line.strip()}')
        break
    print(f'Attempt {attempt+1}: waiting...')
    time.sleep(5)
else:
    print('Decommission may need more time. Check manually:')
    print('  make shell-namenode')
    print('  hdfs dfsadmin -report | grep -A5 datanode3')

In [ ]:
# 4d. Recommission the node (undo decommission)
#     Remove datanode3 from the exclude file and refresh.
print('Recommissioning datanode3...')

# Write empty exclude file to commision all nodes
with open(local_exclude, 'w') as f:
    f.write('')  # empty = no excluded nodes
docker_cp(local_exclude, '/opt/hadoop/etc/hadoop/exclude-hosts')
hadoop('hdfs dfsadmin -refreshNodes')

# Verify
report = hadoop('hdfs dfsadmin -report 2>&1 | grep "Live datanodes"')
print('After recommission:')
print(report)

In [ ]:
# 4e. Clean up temp files
import os
os.remove(local_exclude)

## 5. NameNode High Availability (Conceptual)

The NameNode is HDFS's **single point of failure**. If it goes down, the entire cluster is offline until it's restored. In production, this is unacceptable.

### How HA Works

| Component | Role |
|---|---|
| **Active NameNode** | Handles all client operations (like our single namenode) |
| **Standby NameNode** | Mirrors the Active — takes over on failure |
| **Quorum Journal Manager (QJM, 3+ nodes)** | Shared edit log: both NameNodes read/write here |
| **ZKFailoverController (ZKFC)** | Monitors health, triggers failover via ZooKeeper |

### Failover Flow
1. ZKFC detects Active NN is unhealthy (lost heartbeat)
2. ZKFC in the Standby NN promotes it to Active
3. The new Active NN reads the last transaction from the Journal
4. DataNodes receive the new Active NN identity via heartbeat
5. **Result:** clients experience a brief pause (seconds), not a cluster outage

### What you would configure
```xml
<property><name>dfs.nameservices</name><value>mycluster</value></property>
<property><name>dfs.ha.namenodes.mycluster</name><value>nn1,nn2</value></property>
<property><name>dfs.namenode.rpc-address.mycluster.nn1</name><value>namenode1:8020</value></property>
<property><name>dfs.namenode.rpc-address.mycluster.nn2</name><value>namenode2:8020</value></property>
<property><name>dfs.client.failover.proxy.provider.mycluster</name><value>...ConfiguredFailoverProxyProvider</value></property>
<property><name>dfs.namenode.shared.edits.dir</name><value>qjournal://journal1:8485;journal2:8485;journal3:8485/mycluster</value></property>
```

### Our lab vs production
| Aspect | This lab | Production |
|---|---|---|
| NameNodes | 1 (single point of failure) | 2+ (Active + Standby) |
| Metadata storage | Local disk + fsimage | Shared Journal (QJM) + fsimage |
| Failover | Manual (`make clean` + restart) | Automatic (ZKFC) in < 30s |
| Journal | None (single node) | 3+ JournalNodes on separate hosts |

> **Try it:** Stop the namenode (`docker compose stop namenode`) and observe that all HDFS operations (`client.list('/')`) hang. Now you see why HA matters.

## 6. Summary

| Scenario | What we did | Why it matters |
|---|---|---|
| **Safe Mode** | Entered/left safe mode, observed write rejection | Maintenance, corruption recovery |
| **Block Rebalancing** | Ran `hdfs balancer` after node restart | Even storage utilization, performance |
| **Decommissioning** | Gracefully decommissioned + recommissioned a DataNode | Zero-downtime hardware replacement |
| **NameNode HA** | Conceptual exploration | Eliminates the single point of failure |

### Cleanup
Remove the test data from HDFS and local temp.

In [ ]:
safe_delete(client, TEST_DIR)
import shutil
shutil.rmtree('../temp/ftest', ignore_errors=True)
print('Cleanup complete ✓')